# 01 — Exploration du dataset ShapeNet complet

Ce notebook travaille sur **ton vrai dataset** :
- nuages de points : `ShapeNet_pcs/` (~52 000 fichiers `.ply`, ~70 Go)
- étiquettes : `labeled_dataset.csv` (correspondance id → classe, fourni avec le dataset)

Aucun sous-ensemble artificiel (`subset_metadata_*`) n'est utilisé ici.

> **Note :** `labeled_dataset.csv` n'est pas un sous-échantillon — c'est le fichier officiel
> qui associe chaque `.ply` à sa classe. Sans lui, on ne peut pas étudier la distribution
> des classes ni faire de classification.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from plyfile import PlyData
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LABELS_CSV = PROJECT_ROOT / "labeled_dataset.csv"
POINTCLOUD_ROOT = PROJECT_ROOT / "ShapeNet_pcs"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures" / "dataset_exploration"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
print(f"Projet          : {PROJECT_ROOT}")
print(f"Nuages de points: {POINTCLOUD_ROOT}")
print(f"Étiquettes      : {LABELS_CSV}")

## 1. Vue d'ensemble du dataset

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
ply_files = {path.stem: path for path in POINTCLOUD_ROOT.glob("*.ply")}

labels_df["ply_exists"] = labels_df["id"].isin(ply_files)
dataset_df = labels_df[labels_df["ply_exists"]].copy().reset_index(drop=True)

print(f"Entrées dans labeled_dataset.csv : {len(labels_df):,}")
print(f"Fichiers .ply sur disque           : {len(ply_files):,}")
print(f"Objets exploitables (id + .ply)    : {len(dataset_df):,}")
print(f"Nombre de classes                  : {dataset_df['label'].nunique()}")
print(f"Objets sans .ply                   : {(~labels_df['ply_exists']).sum()}")

dataset_df.head()

## 2. Distribution des classes

In [ ]:
class_counts = dataset_df["label"].value_counts().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

class_counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Nombre d'objets par classe")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Nombre d'objets")
axes[0].tick_params(axis="x", rotation=75)

axes[1].hist(class_counts.values, bins=20, color="coral", edgecolor="white")
axes[1].set_title("Histogramme des effectifs par classe")
axes[1].set_xlabel("Objets par classe")
axes[1].set_ylabel("Nombre de classes")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "class_distribution.png", dpi=150)
plt.show()

print(f"Min / médiane / max par classe : {class_counts.min()} / {class_counts.median():.0f} / {class_counts.max()}")
print(f"Ratio max/min (déséquilibre)   : {class_counts.max() / class_counts.min():.2f}")

In [ ]:
summary = pd.DataFrame({
    "classe": class_counts.index,
    "nombre_objets": class_counts.values,
    "pourcentage": (class_counts.values / len(dataset_df) * 100).round(2),
})
summary

## 3. Statistiques géométriques (échantillon)

On analyse un échantillon aléatoire pour éviter de charger les 50 000+ fichiers.

In [ ]:
SAMPLE_SIZE = 500  # augmenter si tu veux plus de précision (plus lent)


def load_ply_stats(ply_path: Path) -> dict:
    vertex = PlyData.read(str(ply_path))["vertex"]
    names = vertex.data.dtype.names
    pos = np.stack([vertex[axis] for axis in ("x", "y", "z")], axis=1).astype(np.float32)

    if all(channel in names for channel in ("red", "green", "blue")):
        colors = np.stack([vertex[channel] for channel in ("red", "green", "blue")], axis=1)
        color_mean = float(colors.mean())
        color_std = float(colors.std())
    else:
        color_mean = np.nan
        color_std = np.nan

    bbox = pos.max(axis=0) - pos.min(axis=0)
    return {
        "num_points": len(pos),
        "bbox_x": float(bbox[0]),
        "bbox_y": float(bbox[1]),
        "bbox_z": float(bbox[2]),
        "bbox_volume": float(bbox.prod()),
        "color_mean": color_mean,
        "color_std": color_std,
    }


sample_df = dataset_df.sample(n=min(SAMPLE_SIZE, len(dataset_df)), random_state=SEED)
stats_rows = []
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Analyse .ply"):
    ply_path = ply_files[row["id"]]
    stats = load_ply_stats(ply_path)
    stats["id"] = row["id"]
    stats["label"] = row["label"]
    stats_rows.append(stats)

stats_df = pd.DataFrame(stats_rows)
stats_df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(stats_df["num_points"], bins=30, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Nombre de points par objet")

sns.histplot(stats_df["bbox_volume"], bins=30, ax=axes[0, 1], color="seagreen")
axes[0, 1].set_title("Volume de la bounding box")

sns.histplot(stats_df["color_mean"].dropna(), bins=30, ax=axes[1, 0], color="coral")
axes[1, 0].set_title("Moyenne des couleurs RGB")

sns.histplot(stats_df["color_std"].dropna(), bins=30, ax=axes[1, 1], color="mediumpurple")
axes[1, 1].set_title("Écart-type des couleurs RGB")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "geometry_stats.png", dpi=150)
plt.show()

In [ ]:
points_by_class = stats_df.groupby("label")["num_points"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
points_by_class.head(10)

## 4. Visualisation 3D de nuages de points

In [ ]:
def load_point_cloud(object_id: str):
    ply_path = ply_files[object_id]
    vertex = PlyData.read(str(ply_path))["vertex"]
    names = vertex.data.dtype.names
    pos = np.stack([vertex[axis] for axis in ("x", "y", "z")], axis=1).astype(np.float32)
    if all(channel in names for channel in ("red", "green", "blue")):
        colors = np.stack([vertex[channel] for channel in ("red", "green", "blue")], axis=1) / 255.0
    else:
        colors = np.ones_like(pos) * 0.5
    return pos, colors


def plot_point_cloud(pos, colors, title="", ax=None, max_points=4096, point_size=0.5):
    if len(pos) > max_points:
        idx = np.random.choice(len(pos), max_points, replace=False)
        pos, colors = pos[idx], colors[idx]
    if ax is None:
        fig = plt.figure(figsize=(5, 5))
        ax = fig.add_subplot(111, projection="3d")
    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=colors, s=point_size)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    return ax

In [ ]:
# Un objet aléatoire du dataset complet
random_row = dataset_df.sample(1, random_state=SEED).iloc[0]
pos, colors = load_point_cloud(random_row["id"])

plot_point_cloud(pos, colors, title=f"{random_row['label']} — {random_row['id']}")
plt.tight_layout()
plt.show()
print(f"Points : {len(pos):,}")

In [ ]:
# 6 classes différentes — un exemple par classe
chosen_classes = class_counts.index[:6]
fig = plt.figure(figsize=(15, 8))

for idx, class_name in enumerate(chosen_classes, start=1):
    row = dataset_df[dataset_df["label"] == class_name].sample(1, random_state=SEED + idx).iloc[0]
    pos, colors = load_point_cloud(row["id"])
    ax = fig.add_subplot(2, 3, idx, projection="3d")
    plot_point_cloud(pos, colors, title=class_name, ax=ax, point_size=1.0)

plt.suptitle("Exemples de nuages de points par classe (dataset complet)")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sample_pointclouds.png", dpi=150)
plt.show()

## 5. Comparaison géométrie (XYZ) vs apparence (RGB)

Utile pour comprendre la partition verticale (VFL) du projet.

In [ ]:
row = dataset_df.sample(1, random_state=SEED).iloc[0]
pos, colors = load_point_cloud(row["id"])

fig = plt.figure(figsize=(12, 4))
ax1 = fig.add_subplot(131, projection="3d")
ax2 = fig.add_subplot(132, projection="3d")
ax3 = fig.add_subplot(133, projection="3d")

plot_point_cloud(pos, np.ones_like(pos) * 0.6, title="Géométrie (XYZ)", ax=ax1, point_size=1.0)
plot_point_cloud(np.zeros_like(pos), colors, title="Apparence (RGB)", ax=ax2, point_size=1.0)
plot_point_cloud(pos, colors, title="Complet (XYZ + RGB)", ax=ax3, point_size=1.0)

plt.suptitle(f"Entité A vs Entité B — {row['label']}")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "xyz_vs_rgb.png", dpi=150)
plt.show()

## 6. Synthèse pour le rapport

In [ ]:
synthesis = {
    "total_objects": int(len(dataset_df)),
    "total_classes": int(dataset_df["label"].nunique()),
    "total_ply_files": int(len(ply_files)),
    "min_objects_per_class": int(class_counts.min()),
    "max_objects_per_class": int(class_counts.max()),
    "median_objects_per_class": float(class_counts.median()),
    "sample_size_for_stats": int(len(stats_df)),
    "mean_num_points": float(stats_df["num_points"].mean()),
    "min_num_points": int(stats_df["num_points"].min()),
    "max_num_points": int(stats_df["num_points"].max()),
}

for key, value in synthesis.items():
    print(f"{key:30s}: {value}")

pd.Series(synthesis).to_csv(FIGURE_DIR / "dataset_synthesis.csv")
print(f"\nFigures sauvegardées dans : {FIGURE_DIR}")